# 3 · One controlled start–shutter–standby session

Practice the complete command order for a single deliberate session: verify preconditions, set power, enable, open and close the safety shutter once, then verify standby.

**Self-contained notebook · simulator default · optional human-operated hardware**

From the repository root, activate your virtual environment, install with `python -m pip install -e ".[tutorials]"`, then launch `python -m jupyterlab examples/tutorials`. Select this environment's Python kernel. Run cells top to bottom with Shift+Enter, or choose **Restart Kernel and Run All Cells**. See [setup and troubleshooting](README.md#start-here).

Every demonstration and hardware helper is defined below. The notebook imports only the controller library and Python's standard library; no tutorial script is loaded. Run cells from top to bottom.

[All four tutorials](README.md)

## 1. Command sequence

Simulator fixture `set_key(True)` (no wire command) → `status()` → `?D1SS, ?ESS, ?VSS` → `?FH` → `P=0.2500, ?SP` → `L=1, ?L, ?F, ?S` → `S=1, ?S, ?F, ?P` → `S=0, ?S` → `L=0, ?L`.

`status()` is the fourteen-query sequence described in lesson 1 and the [tutorial guide](README.md#common-command-vocabulary). `L=1` also resets faults and clears history, so evidence is read first. The internal shutter is a **safety shutter**, never an experiment modulator; there is no cycling loop.

The controller supplies CR/LF framing and waits for each reply. Queries start with `?`; writes use `=`. No `OK` acknowledgment is assumed. Source: supplied [Verdi manual](../../verdi.manual_v5.pdf), Tables 5-1, 5-3 and 5-4; [protocol mapping](../../docs/PROTOCOL.md).

## 2. Import the controller API

Imports do not discover ports, open connections or send commands.

In [ ]:
"""Tutorial 3: one explicit enable, safety-shutter opening, closure and standby."""

from coherent_verdi import SimulatedVerdi, VerdiController

## 3. Define the application steps

Each cell defines one small function. The complete sequence is run in section 4.

### 3.1 Check readiness and preserve fault history

All starting-state checks must pass before a write. Fault history is read before `L=1` can clear it.

Require diode, LBO, etalon and vanadate temperature servos to be LOCKED. On hardware, the key-ON/STANDBY starting state needs a previously approved RS-232 standby override; turning the physical key ON can start the laser. The simulator key fixture is not that physical action.

In [ ]:
def check_ready(laser: VerdiController) -> None:
    before = laser.status()
    if before["laser_state"] != 0 or before["shutter_open"]:
        raise RuntimeError("Start from operator-confirmed STANDBY with shutter closed.")
    if not before["keyswitch_on"] or before["lbo_servo"] != 1 or before["faults"]:
        raise RuntimeError("Require key ON, LBO LOCKED and no active faults; do not auto-enable.")
    for query in ("?D1SS", "?ESS", "?VSS"):
        if laser.read(query) != 1:
            raise RuntimeError(f"Require all temperature servos LOCKED; {query} is not ready.")
    print(f"Fault history before enable: {laser.read_faults(history=True)}")

### 3.2 Set power and enable

Verify the setpoint first. After enabling, verify ON, no active faults and a still-closed shutter.

In [ ]:
def enable_at_power(laser: VerdiController, target_w: float) -> None:
    laser.set_power_w(target_w)
    if laser.read("?SP") != float(f"{target_w:.4f}"):
        raise RuntimeError("Setpoint readback mismatch; enable was not requested.")
    laser.start()  # L=1 also resets faults and clears their history.
    if laser.read_laser_state() != 1 or laser.read_faults():
        raise RuntimeError("Enable did not produce ON without faults.")
    if laser.read("?S") != 0:
        raise RuntimeError("Shutter unexpectedly open after enable.")
    print("Laser ON; shutter still closed.")

### 3.3 Open once and read power

This performs one safety-shutter opening and one immediate sample. It does not wait for optical settling.

In [ ]:
def sample_with_shutter_open(laser: VerdiController) -> float:
    laser.set_shutter(open=True)
    if laser.read("?S") != 1 or laser.read_faults():
        raise RuntimeError("Open-shutter state was not verified without faults.")
    power_w = laser.read_power_w()
    print(f"Shutter open; reported power: {power_w:.3f} W")
    return power_w

### 3.4 Close the shutter and verify standby

Normal completion requires both commands and their readbacks. Releasing communication does not perform these actions.

In [ ]:
def close_and_standby(laser: VerdiController) -> None:
    laser.set_shutter(open=False)
    if laser.read("?S") != 0:
        raise RuntimeError("Shutter closure was not confirmed.")
    laser.stop()
    if laser.read_laser_state() != 0:
        raise RuntimeError("STANDBY was not confirmed.")
    print("Shutter closed; STANDBY confirmed. Setpoint retained for inspection.")

### 3.5 Put the steps in order

The small functions above make the full sequence visible. Any failed step or interruption stops subsequent commands; no blind cleanup or retry occurs.

In [ ]:
def controlled_session(laser: VerdiController, target_w: float) -> float:
    try:
        check_ready(laser)
        enable_at_power(laser, target_w)
        power_w = sample_with_shutter_open(laser)
        close_and_standby(laser)
        return power_w
    except BaseException:
        # Also covers Ctrl+C. A lost reply may mean the command already executed.
        print("STOP: state may be unknown. No retries or blind cleanup commands.")
        print("Use the physical abort procedure; disconnect() is not laser shutdown.")
        raise

## 4. Run the simulator

The fixture setup below is synthetic. The exercise target of 0.25 W and ceiling of 0.5 W are not physical safety limits.

### 4.1 Prepare the simulator demonstration

The function below owns the simulator and its controller lifetime. Each call starts a fresh exercise and leaves no worker or open session.

In [ ]:
def simulate_controlled_session() -> None:
    laser = SimulatedVerdi("V5", allow_writes=True, power_limit_w=0.5)  # Warm fixture.
    laser.set_key(True)
    with laser:
        controlled_session(laser, 0.25)
    print("Connection released after the verified normal-completion sequence.")

### 4.2 Execute the demonstration

Run this short cell to call the functions just defined. Rerun it to begin with a fresh simulator.

In [ ]:
simulate_controlled_session()

## 5. Expected simulator result

```text
Fault history before enable: ()
Laser ON; shutter still closed.
Shutter open; reported power: 0.250 W
Shutter closed; STANDBY confirmed. Setpoint retained for inspection.
Connection released after the verified normal-completion sequence.
```
The setpoint remains 0.2500 W. STANDBY is not a complete heater/cooling shutdown. Simulator power changes instantly; real ramp time, idle power and shutter-closed `?P` behavior remain unverified.

## 6. Try one small change

Change only `laser.set_key(True)` to `laser.set_key(False)` and rerun the simulator example cell. Expect a `RuntimeError` **before any write** plus the STOP message. Restore `True` and rerun the simulator example for the nominal result. This intentional exercise fails the cell; the original notebook runs without unexpected errors.

## 7. Optional real-hardware session for a human operator

Complete the [hardware review procedure](../../HARDWARE_VALIDATION.md) first. Install `python -m pip install -e ".[tutorials,serial]"` in this environment. The hardware functions below are fully visible and call the application functions in section 3 directly.

Fill in the actual model, native port and matching baud. Supply the approved target and ceiling in W; no simulator power default is reused. Confirm cooling, warmup, interlocks, beam conditions and the physical abort procedure. Keep `RUN_HARDWARE=False` for ordinary Run All.

After you enable it, **CONNECT** permits the selected connection and one `?SV` query. Verify the reported device and version. **RUN** permits the displayed lesson. Any other answer cancels. These prompts confirm intent, not site approval.

### 7.1 Import serial interfaces

These imports alone perform no I/O. `contextmanager` lets a `with` block release the connection even on an exception.

In [ ]:
from collections.abc import Iterator
from contextlib import contextmanager

### 7.2 Validate the power configuration

Validation happens before connection. Read-only sessions reject power settings; write sessions require an explicit approved ceiling and target.

In [ ]:
def validate_power(
    model: str, target_w: float | None, power_limit_w: float | None, *, allow_writes: bool
) -> None:
    """Require site values before a physical write; simulator defaults are not limits."""
    if model not in ("V2", "V5", "V6"):
        raise ValueError("model must be V2, V5 or V6")
    if not allow_writes:
        if target_w is not None or power_limit_w is not None:
            raise ValueError("Read-only lessons do not accept power settings")
        return
    if (
        isinstance(power_limit_w, bool)
        or not isinstance(power_limit_w, (int, float))
        or not 0 <= power_limit_w <= float(model[1:])
    ):
        raise ValueError("Supply the approved power_limit_w within the model rating")
    if (
        isinstance(target_w, bool)
        or not isinstance(target_w, (int, float))
        or not 0 <= target_w <= power_limit_w
        or float(f"{target_w:.4f}") > power_limit_w
    ):
        raise ValueError("target_w must be finite, nonnegative and within the approved ceiling")

### 7.3 Connect, identify and release

This helper prompts before opening, sends `?SV` first and closes communication when the `with` block ends. An error stops further commands. **Connection close is not physical shutdown**; use the operator's abort procedure when state is uncertain.

In [ ]:
@contextmanager
def hardware_connection(laser: VerdiController) -> Iterator[VerdiController | None]:
    """Operator confirmation, passive connect, one ?SV, and deterministic disconnect."""
    if input("With candidate/connection approval recorded, type CONNECT: ").strip() != "CONNECT":
        print("Cancelled before opening the port.")
        yield None
        return
    try:
        with laser:
            print(f"Reported software: {laser.read('?SV')}")
            yield laser
    except BaseException:
        print("STOP: state may be unknown. No retries or blind cleanup writes.")
        print("Use the operator's approved physical abort procedure.")
        raise
    finally:
        print("Connection released; disconnect does not shut down the laser.")

### 7.4 Enter the operator's settings

Nothing connects when you run this configuration cell. Set `RUN_HARDWARE=True` only for an attended, approved session. Restore False afterwards.

The manual does not specify the no-active-fault reply for `?F`. Keep `ACTIVE_FAULT_CLEAR_REPLY=None` until Stage 1 records its exact meaning for this firmware. Then enter the verified text here; otherwise a clear-looking response stops the physical session. `SYSTEM OK` is documented for `?FH` only. Simulator defaults use their explicit fixture convention.

In [ ]:
RUN_HARDWARE = False  # Change only for an approved, attended physical session.
HARDWARE_PORT = None  # Set to the operator-confirmed native port, e.g. "COM3".
HARDWARE_MODEL = None  # Set to 'V2', 'V5' or 'V6' after checking the label.
HARDWARE_BAUDRATE = None  # Set to the actual front-panel baud rate.
HARDWARE_TIMEOUT_S = 1.0  # Adjust to the validated transaction deadline.
TARGET_W = None  # Operator-approved requested setpoint, in W.
POWER_LIMIT_W = None  # Operator-approved ceiling, in W.
ACTIVE_FAULT_CLEAR_REPLY = None  # Set only after Stage 1 verifies the exact ?F clear text.

### 7.5 Run the visible hardware sequence

Calls the complete `controlled_session` from section 3. It enables, opens once, reads power, closes the shutter and verifies standby. The target remains stored.

The call completes in one cell; no connection waits between cells. A new run prompts again.

In [ ]:
if RUN_HARDWARE:
    model = HARDWARE_MODEL
    validate_power(model, TARGET_W, POWER_LIMIT_W, allow_writes=True)
    controller = VerdiController(
        HARDWARE_PORT,
        model=model,
        baudrate=HARDWARE_BAUDRATE,
        timeout_s=HARDWARE_TIMEOUT_S,
        active_fault_clear_reply=ACTIVE_FAULT_CLEAR_REPLY,
        allow_writes=True,
        power_limit_w=POWER_LIMIT_W,
    )
    print(f"HARDWARE: {model}, port={HARDWARE_PORT}, baud={HARDWARE_BAUDRATE}")
    print("Plan: set power, enable, open once, read, close and verify STANDBY.")
    print(f"Target: {TARGET_W} W; ceiling: {POWER_LIMIT_W} W.")
    with hardware_connection(controller) as laser:
        if laser is not None:
            if input("Verify device/version and approved scope; type RUN: ").strip() == "RUN":
                controlled_session(laser, TARGET_W)
            else:
                print("Cancelled after ?SV; no lesson commands sent.")
else:
    print("Hardware section skipped. Set explicit operator configuration to use it.")

## 8. What this establishes

Simulator runs verify software behavior, not physical response or calibration. A status sample is sequential, not an interlock or proof of a safe beam path. Review the [simulator assumptions](../../docs/SIMULATOR.md) and [operator guide](README.md#human-operated-hardware) before physical use.